### 1.1.7. Sets of Numbers

AMPL sets may contain **numbers** as members, not only strings. Numeric members participate in
arithmetic — a dummy index drawn from a numeric set may appear in arithmetic expressions such
as `t - 1` or `2 * t`. This distinguishes numeric sets from symbolic (string) sets and makes
them the natural representation for time periods, integer indices, frequency levels, and any other
quantity whose members carry mathematical meaning beyond mere labels.

The most important syntactic feature for numeric sets is the **arithmetic range expression**
`e1 .. e2 by e3`, which constructs a finite set of evenly-spaced numbers without requiring
explicit enumeration. Arithmetic ranges may appear anywhere a set expression is valid:
in a set declaration, as an inline index set, as an operand to set operators, or in a summation.

In [ ]:
# Arithmetic range syntax (formal grammar)
#
#   expr1 .. expr2 [ by expr3 ]
#
#   expr1  — lower bound (any arithmetic expression)
#   expr2  — upper bound (any arithmetic expression)
#   expr3  — step / interval (any arithmetic expression; default is 1)
#
# Semantics (AMPL Reference Manual, A.4):
#   The set contains the numbers
#       expr1,  expr1 + expr3,  expr1 + 2*expr3, ...
#   up to and including the largest value <= expr2.
#   Each value is rounded to the nearest representable floating-point number.
#   The 'by' step may be negative, producing a descending sequence.
#
# Example instantiations:
1 .. 6              # {1, 2, 3, 4, 5, 6}                   — step 1 (default)
2010 .. 2040 by 5   # {2010, 2015, 2020, 2025, 2030, 2035, 2040}
0 .. 1 by 0.25      # {0, 0.25, 0.5, 0.75, 1.0}            — non-integer step
10 .. 1 by -3       # {10, 7, 4, 1}                         — descending step

#### Conceptual Explanation

**Numeric members and arithmetic.** In AMPL, a literal number is any sequence of digits with
an optional sign, optional decimal point, and optional exponent (`d`, `D`, `e`, or `E`). The
number `100` and `1E+2` denote the same set member; the number `1` and the string `"1"` are
distinct. All representations that evaluate to the same floating-point value are treated as
identical members.

A set of numbers can be specified by explicit enumeration or by an arithmetic range expression.
The range `e1 .. e2 by e3` (Reference Manual, §A.4) produces:

$$\left\{\, e_1 + k \cdot e_3 \;:\; k = 0, 1, 2, \ldots, \left\lfloor\frac{e_2 - e_1}{e_3}\right\rfloor \,\right\}$$

rounded to the nearest representable floating-point number. The default step is 1, so `1..T`
is shorthand for `1..T by 1`.

**Integer vs. real-valued members.** AMPL makes no type-level distinction between integer and
real-valued numeric sets at the language level — both are sets of floating-point numbers.
In practice, the distinction is important:

- *Integer-valued sets* (step = 1, or any integer step, with integer bounds) store members
  exactly for magnitudes up to $2^{53}$ under IEEE 754 arithmetic.
- *Real-valued sets* (non-integer bounds or step) accumulate floating-point rounding error
  per step. Membership tests against such a set may fail unexpectedly if the queried value
  does not round to exactly the same floating-point number as the stored member.

**Arithmetic on dummy indices.** Because members of a numeric set are numbers, a dummy
index drawn from the set may appear in arithmetic expressions. For example, if `t in 1..T`,
then `t - 1` is a valid arithmetic expression referring to the previous period's index. This
is the mechanism behind the multi-period inventory balance constraint. By contrast, if `t`
were a string (e.g., `t in WEEKS` where `WEEKS` is a set of strings), then `t - 1` would be
a type error.

#### Arithmetic Range Attributes

The table below documents all components of the `..` range expression.

| Component | Required | Type | Description |
|---|---|---|---|
| `e1` (lower bound) | Yes | Arithmetic expression | First member of the set |
| `e2` (upper bound) | Yes | Arithmetic expression | Members do not exceed this value |
| `by e3` (step) | No | Arithmetic expression (non-zero) | Spacing between consecutive members; default is 1 |

All three components may be **any arithmetic expression**, including references to declared
parameters. The set is re-evaluated whenever a dependency changes. Each component is evaluated
to a floating-point number before the sequence is constructed.

**Operator precedence.** The `..` operator has the highest precedence among set operators
(precedence 11 in AMPL Reference Manual Table A-1, above `cross` at 10, `inter` at 9, and
`union`/`diff`/`symdiff` at 8). Consequently, `1..T union {0}` parses as `(1..T) union {0}`,
which is the intended meaning. Parentheses may always be used to clarify grouping.

**Predefined infinite sets.** AMPL provides two predefined numeric sets for use in `within`
constraints on parameters and variables:
- `Integers` — the set of all integers, ordered numerically
- `Reals` — the set of all floating-point numbers, ordered numerically

These are infinite sets and cannot be enumerated directly; they are used only as domains for
`within` constraints (e.g., `param T integer;` is equivalent to `param T in Integers;`).

#### Example 1 — Arithmetic Range Basics

In [ ]:
# Integer range: implicit step of 1
set PERIODS = 1 .. 6;          # {1, 2, 3, 4, 5, 6}

# Integer range with explicit step
set YEARS = 2010 .. 2040 by 5; # {2010, 2015, 2020, 2025, 2030, 2035, 2040}

# Descending range
set COUNTDOWN = 10 .. 1 by -3; # {10, 7, 4, 1}

# Range parameterized by data — the recommended practice for changeable bounds
param T > 0 integer;            # number of periods; value comes from data
param avail {1..T} >= 0;        # hours available indexed over 1..T

# Range with parameterized bounds and step
param start    integer;
param end_yr   > start integer;
param interval > 0 integer;
set FLEX_YEARS = start .. end_yr by interval;   # all three values from data

# Range used inline (not assigned to a named set)
param market {PROD, 1..T} >= 0; # no named set needed — range used directly
set PROD;                        # referenced above; declared here for completeness

#### Example 2 — Numeric Sets vs. String Sets: Membership Test and Arithmetic

In [ ]:
# Membership test with a numeric literal
set T_SET = 1 .. 10;
set PRODS;

# Membership test: numeric literal on the left of 'in'
param has_5    = if 5    in T_SET  then 1 else 0;  # 1 (5 is a member)
param has_11   = if 11   in T_SET  then 1 else 0;  # 0 (11 exceeds upper bound)

# Membership test: string literal on the left of 'in'
param has_bands = if 'bands' in PRODS then 1 else 0;

# Arithmetic on a numeric dummy index is valid
param T > 0 integer;
param inv0  {PRODS} >= 0;
var Make {PRODS, 1..T} >= 0;
var Inv  {PRODS, 0..T} >= 0;    # note: 0..T includes period 0 for initial inventory
var Sell {PRODS, 1..T} >= 0;

# t-1 is a valid arithmetic expression because t is a number from 1..T
subject to Balance {p in PRODS, t in 1..T}:
    Make[p,t] + Inv[p,t-1] = Sell[p,t] + Inv[p,t];

# Initial inventory (period 0)
subject to Init_Inv {p in PRODS}: Inv[p,0] = inv0[p];

# Arithmetic on a STRING dummy index would be REJECTED by AMPL:
# set WEEKS_STR ordered;  # string-named weeks
# var Inv2 {PRODS, WEEKS_STR} >= 0;
# subject to Bad_Balance {p in PRODS, t in WEEKS_STR}:
#     Inv2[p, t-1] ...;  # ERROR: t is a string; t-1 is a type error

#### Example 3 — Real-Valued Numeric Sets

Set members need not be integers. AMPL stores each numerical set member as the nearest
representable floating-point number. Non-integer steps can produce rounding errors that
cause membership tests to fail unexpectedly (see Pitfall below).

In [ ]:
# Real-valued set with a fractional step
# Members stored as nearest floating-point values:
#   {0.0, 0.25, 0.5, 0.75, 1.0}
# 0.25 is exactly representable in IEEE 754 binary64, so this case is safe.
set RATES = 0.0 .. 1.0 by 0.25;

# Non-integer step with non-exact representation — avoid for membership tests
# Members nominally {0, 0.1, 0.2, ..., 1.0} but stored as rounded float values.
# Membership test '0.3 in S' may return 0 due to rounding.
# set S = 0.0 .. 1.0 by 0.1;   # use with caution — see Pitfall

# Safe pattern: use integer indexing and scale in expressions
set IDX = 0 .. 10;              # {0, 1, 2, ..., 10} — stored exactly
param discount_rate {i in IDX} = i / 10;   # compute rate as 0.0, 0.1, ..., 1.0

# Using RATES as an index for a parameter
param output_at {RATES} >= 0;   # output level at each rate in the range

# Cardinality of RATES
param n_rates = card(RATES);    # = 5

#### Example 4 — Multiperiod Production Model: Numeric Sets as Index Sets

This is the canonical use of numeric sets in AMPL: a parameter `T` defines the time horizon,
and the inline expressions `1..T` and `0..T` are used as index sets for parameters, variables,
and constraints. This is drawn directly from the multiperiod steel production model
(Fourer, Gay, Kernighan, Figure 4-4).

In [ ]:
# --- steelT.mod (simplified multiperiod production model) ---

set PROD;               # products
param T > 0 integer;    # number of weeks (from data)

param rate     {PROD}        > 0;   # tons per hour produced
param inv0     {PROD}       >= 0;   # initial inventory
param avail    {1..T}       >= 0;   # hours available in each week
param market   {PROD, 1..T} >= 0;   # market limit on tons sold in each week
param prodcost {PROD}       >= 0;   # cost per ton produced
param invcost  {PROD}       >= 0;   # carrying cost per ton of inventory
param revenue  {PROD, 1..T} >= 0;   # revenue per ton sold

var Make {PROD, 1..T}                    >= 0;   # tons produced
var Inv  {PROD, 0..T}                    >= 0;   # tons in inventory (0 = initial)
var Sell {p in PROD, t in 1..T}          >= 0,
                                         <= market[p,t];

maximize Total_Profit:
    sum {p in PROD, t in 1..T}
        (revenue[p,t] * Sell[p,t]
         - prodcost[p] * Make[p,t]
         - invcost[p]  * Inv[p,t]);

subject to Time {t in 1..T}:
    sum {p in PROD} (1/rate[p]) * Make[p,t] <= avail[t];

subject to Init_Inv {p in PROD}: Inv[p,0] = inv0[p];

# t-1 is valid arithmetic because t is drawn from the numeric set 1..T
subject to Balance {p in PROD, t in 1..T}:
    Make[p,t] + Inv[p,t-1] = Sell[p,t] + Inv[p,t];

#### Example 5 — Numeric Set Data in `.dat` Files vs. Symbolic Definition

Numeric sets can be populated from data files just like string sets. The `.dat` syntax
uses unquoted numbers as set members.

In [ ]:
# --- model file ---

# Option A: define symbolically using an arithmetic range (re-evaluated from param data)
param start    integer;
param end_yr   > start integer;
param interval > 0 integer;
set YEARS_SYM = start .. end_yr by interval;   # derived — updated when params change

# Option B: declare only — membership comes entirely from the data file
set YEARS_DATA;   # AMPL treats members as numbers if the data supplies numbers

# Both approaches allow the same downstream use:
param output {YEARS_SYM} >= 0;
param price  {YEARS_DATA} >= 0;

# --- data file (.dat) for Option A ---
# param start    := 2010;
# param end_yr   := 2040;
# param interval := 5;

# --- data file (.dat) for Option B ---
# data;
# set YEARS_DATA := 2010 2015 2020 2025 2030 2035 2040;
#
# Numbers in .dat files are unquoted; quoted values would be strings.
# '2010' (with quotes) and 2010 (without quotes) are DIFFERENT set members.

#### `card()` on Numeric Sets

The built-in function `card(S)` returns the number of members in any finite set `S`,
including numeric sets and arithmetic ranges (AMPL Reference Manual, §A.6.1). Its argument
may be any set expression — a named set, an arithmetic range, a set operation result,
or an indexing expression.

In [ ]:
# card() on a named numeric set
param T > 0 integer;
set PERIODS = 1 .. T;
param n_periods = card(PERIODS);   # = T

# card() on an inline arithmetic range (argument is a set expression)
param n_range = card(2010 .. 2040 by 5);   # = 7

# card() on a set operation result
set Y1 = 2010 .. 2040 by 5;   # {2010, 2015, 2020, 2025, 2030, 2035, 2040}
set Y2 = 2020 .. 2045 by 5;   # {2020, 2025, 2030, 2035, 2040, 2045}
param n_union = card(Y1 union Y2);  # = 8
param n_inter = card(Y1 inter Y2);  # = 5

# card() used to parameterize another model component
param half_T = card(1 .. T) / 2;   # useful for splitting a horizon

# card() with an indexing-expression argument (parentheses may be omitted per Reference Manual)
param n_even = card {t in 1..T: t mod 2 = 0};  # number of even periods

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Parameterize arithmetic range bounds rather than embedding literals. Write
<code>1..T</code> with <code>param T &gt; 0 integer;</code> supplied from data, rather than
<code>1..52</code> hard-coded in the model. This keeps the model stable across problem
instances that differ only in time horizon, and avoids the need to edit model code when
the planning horizon changes.
</div>

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Use integer-valued arithmetic ranges (integer bounds, integer step) whenever the set
members need to satisfy membership tests or be used in comparisons. Integers are
represented exactly up to $2^{53}$ under IEEE 754, so <code>5 in 1..10</code> always
returns true. Avoid fractional steps such as <code>0.0..1.0 by 0.1</code> in any context
where exact membership is required; instead, index over an integer set and scale
arithmetically (e.g., <code>set IDX = 0..10; param rate {i in IDX} = i/10;</code>).
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Fractional steps in arithmetic ranges accumulate floating-point rounding error. AMPL
stores each member as the nearest representable floating-point number. The range
<code>0.0..1.0 by 0.1</code> does not reliably contain <code>0.3</code> as a member,
because <code>0.0 + 3*0.1</code> evaluates to <code>0.30000000000000004</code> in IEEE 754
binary64, which is a different floating-point value from the literal <code>0.3</code>.
The AMPL book (Chapter 5) explicitly warns: <em>"It is unwise to use fractional numbers
in sets, if your model relies on set members having precise values."</em>
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Quoted numbers and unquoted numbers are distinct in AMPL. In a data file, writing
<code>set YEARS := 2010 2015 2020;</code> produces a numeric set with members
<code>2010</code>, <code>2015</code>, <code>2020</code>. Writing
<code>set YEARS := '2010' '2015' '2020';</code> produces a symbolic (string) set.
A membership test <code>2010 in YEARS</code> returns true only for the numeric set;
for the string set it returns false because <code>2010</code> (a number) and
<code>'2010'</code> (a string) are different AMPL values.
</div>

#### amplpy Equivalents

When using amplpy, numeric sets can be assigned from Python `range` objects, lists of
integers, or lists of floats. The table below shows all relevant patterns.

| AMPL operation | amplpy equivalent |
|---|---|
| `set T = 1..10;` (symbolic) | Define in model via `ampl.eval("set T = 1..10;")` |
| `set YEARS_DATA;` + data `:= 2010 2015 ...` | `ampl.set["YEARS_DATA"] = list(range(2010, 2045, 5))` |
| `ampl.set["S"] = [...]` | Pass a Python list of int or float values |
| `ampl.get_set("S").set_values([...])` | Equivalent explicit form |
| `param T := 10;` (scalar driving a range) | `ampl.param["T"] = 10` |
| `card(S)` in model | `ampl.get_set("S").size()` or `ampl.get_value("card(S)")` |
| `5 in T` (membership test) | `ampl.get_value("5 in T")` returns 1.0 or 0.0 |
| Read members of a numeric set | `ampl.get_set("S").members()` — returns a Python list |
| `ampl.set["S"].members()` | Returns list of Python int or float values |

In [ ]:
# amplpy: assigning numeric sets from Python

# Pattern 1: drive a symbolic range through a scalar parameter
# Model declares: param T > 0 integer; set PERIODS = 1..T;
ampl.param["T"] = 10          # PERIODS becomes {1, 2, ..., 10} automatically

# Pattern 2: property assignment from a Python range
# Model declares: set YEARS_DATA;
ampl.eval("set YEARS_DATA;")
ampl.set["YEARS_DATA"] = list(range(2010, 2045, 5))  # [2010, 2015, ..., 2040]

# Pattern 3: explicit set_values from a list of integers
ampl.eval("set IDX;")
ampl.get_set("IDX").set_values([1, 2, 3, 4, 5])

# Pattern 4: assigning a set of floats (use with caution — see Pitfall)
ampl.eval("set RATES;")
ampl.set["RATES"] = [0.0, 0.25, 0.5, 0.75, 1.0]  # exact quarters — safe

# Reading members back into Python
members = ampl.get_set("YEARS_DATA").members()   # returns a list of Python ints/floats
n       = ampl.get_set("YEARS_DATA").size()      # cardinality

# Evaluating membership and card() via AMPL expressions
in_set  = ampl.get_value("2020 in YEARS_DATA")   # 1.0 if member, 0.0 otherwise
n_expr  = ampl.get_value("card(YEARS_DATA)")     # same as .size()

#### Edge Cases

**Edge case 1: Empty arithmetic range.**

If `e1 > e2` with a positive step (or `e1 < e2` with a negative step), the range is
empty. AMPL does not raise an error; the result is the empty set `{}`. Any indexing over
an empty range generates zero instances of the indexed component — no constraints, no
summation terms. This is well-defined behavior, not an error.

In [ ]:
# Empty range: lower bound exceeds upper bound with positive step
param T_begin integer;
param T_end   integer >= T_begin;

# If T_begin = T_end, the range 1..(T_end - T_begin) = 1..0, which is empty
# Any constraints indexed over an empty range generate zero instances:
param avail2 {1..T_end} >= 0;
set PROD2;
var Make2 {PROD2, T_begin..T_end} >= 0;

# If T_begin > T_end, Make2 is declared over an empty set — still valid in AMPL.
# card(T_begin..T_end) = 0 in that case.

# card() on empty range
param n_empty = card(5 .. 1);   # = 0 (step defaults to 1, lower > upper)

**Edge case 2: Arithmetic range used as an operand to set operators.**

Arithmetic ranges are valid set expressions and may be used directly as operands to
`union`, `inter`, `diff`, `symdiff`, and `cross` without first assigning them to a named
set. Inline use avoids the need to declare a separate set name.

In [ ]:
set Y1 = 2010 .. 2040 by 5;    # {2010, 2015, 2020, 2025, 2030, 2035, 2040}
set Y2 = 2020 .. 2045 by 5;    # {2020, 2025, 2030, 2035, 2040, 2045}

# Set operations on named numeric ranges
display Y1 union Y2;    # {2010, 2015, 2020, 2025, 2030, 2035, 2040, 2045}
display Y1 inter Y2;    # {2020, 2025, 2030, 2035, 2040}
display Y1 diff  Y2;    # {2010, 2015}
display Y1 symdiff Y2;  # {2010, 2015, 2045}

# Inline range expression as an operand
display Y1 union {2045};            # appending a singleton: {2010, ..., 2040, 2045}
display 2020..2060 by 5 symdiff (Y1 union Y2);  # {2050, 2055, 2060, 2010, 2015}

# Range in a summation — no named set needed
param T > 0 integer;
param cost_per_period {1..T} >= 0;
param total_cost = sum {t in 1..T} cost_per_period[t];   # inline range in sum

**Edge case 3: Membership test with a value computed by arithmetic.**

A membership test `e in S` where `e` is an arithmetic expression is valid when `S` is a
numeric set and `e` evaluates to a number. The test checks whether the computed value
is one of the stored (rounded) floating-point members of `S`. For integer-valued sets
and integer-valued expressions this is exact. For real-valued sets and expressions
involving division, the result depends on floating-point representation.

In [ ]:
set T_SET = 1 .. 10;

# Membership test with a numeric literal: always exact for integers
param chk1 = if 5    in T_SET then 1 else 0;   # 1
param chk2 = if 10   in T_SET then 1 else 0;   # 1 (upper bound is included)
param chk3 = if 11   in T_SET then 1 else 0;   # 0 (exceeds upper bound)
param chk4 = if 0    in T_SET then 1 else 0;   # 0 (below lower bound)

# Membership test with an arithmetic expression
param T > 0 integer;
param half = T / 2;                             # may not be integer if T is odd
param mid_in = if (T div 2) in T_SET then 1 else 0;   # 'div' gives integer quotient

# Membership test with a set of strings vs. set of numbers — distinct types
set STR_SET = {'2010', '2015', '2020'};   # string set
set NUM_SET = {2010, 2015, 2020};         # numeric set

# 2010 (number) in STR_SET is FALSE — number and string are different AMPL values
# '2010' (string) in NUM_SET is FALSE — same reason
# 2010 (number) in NUM_SET is TRUE
param q1 = if 2010   in NUM_SET then 1 else 0;   # 1
param q2 = if '2010' in STR_SET then 1 else 0;   # 1

**Edge case 4: Large arithmetic ranges and memory.**

AMPL materializes the members of an arithmetic range as a finite set in memory. For
very large ranges such as `1..1e9`, AMPL will attempt to allocate storage for all
$10^9$ members, which typically exhausts available memory. The range expression is not
lazy — it is evaluated to a concrete set at model-generation time.

In [ ]:
# Safe: T from data, reasonably sized (e.g., T = 52 weeks or T = 365 days)
param T > 0 integer;          # value from data; model code is unchanged
set PERIODS = 1..T;           # materialized at model generation — OK if T is small

# Unsafe for large T: 1..1e9 would attempt to store 10^9 members
# set HUGE = 1..1e9;          # likely causes out-of-memory error at generation time

# Better alternative for large continuous-time problems:
# Work with representative sample periods rather than enumerating all time points.

# card() provides a fast bound check without materializing an extra set
check: card(PERIODS) <= 10000;   # guard against accidentally large time horizons

#### Cross-References

- See also: [1.1.1. Set Declaration vs. Definition](<1_1_1 Set Declaration vs. Definition.ipynb>) — for the `=` vs. `:=` vs. `default` distinction that applies equally to numeric sets
- See also: [1.1.4. Set Operations](<1_1_4 set_operations.ipynb>) — for `union`, `inter`, `diff`, `symdiff`, and `cross` on numeric ranges
- See also: [1.1.5. Unordered Sets](<1_1_5 unordered_sets.ipynb>) — for the distinction between unordered and ordered numeric sets, and when `prev`/`next`/`ord` apply
- AMPL book Chapter 5.2 (Sets of Numbers) — primary source for arithmetic range semantics
- AMPL Reference Manual §A.4 (Expressions, `..` operator) — formal grammar and semantics
- AMPL Reference Manual §A.6.1 (`card`, `arity`, `indexarity` functions)